## Learning Objectives

* Filter and sort data using Pandas
* Show how `.loc[]` works in Pandas
* Understand the Pandas `.groupby()` function for aggregations
* Show how Pandas uses NumPy
* Do basic NumPy operations

In [ ]:
# imports


In [ ]:
# Read in the diamonds dataset



In [ ]:
# check for missing values


In [ ]:
# checking .info()


## Accessing columns and rows

We've seen how to access individual and multiplie columns with the following syntax:

```python
df['col1']

df[['col1', 'col2']]
```

Pandas has a more preferred method of accessing data inside dataframes, `.loc[]`, especially if you're going to modify data down the road.

[.loc docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)

In [ ]:
# grab the 'carat' column


In [ ]:
# We can grab multiple columns by passing a list after the , inside .loc[]



In [ ]:
# We can also slice our dataframe like a list



---

## Filtering and Sorting

We've see how to do some filtering before, we'll also want to sort our dataframes.

We don't always need to use a full dataset. It's a very common task to parse down the data to only the pieces we need.

In [ ]:
# We saw .describe() used on our entire df before, we can call it on individual cols as well


In [ ]:
# How could I grab the row with the max price?


In [ ]:
# Let's filter to the diamonds that are higher than $10k


In [ ]:
# each of our filters does a boolean check, then only returns the rows where that condition is True


In [ ]:
# We'll often save our filter condition as a new variable then pass that through .loc[]
# this is called masking



## Multiple filters

We can filter on multiple conditions! That is very common!

This is where `.loc[]` can get weird.

Use `df['col']` to build conditions

Use `.loc[]` to apply them.

In [ ]:
# filter on diamonds less than 5 carats and are ideal cut



### *Aside* 

Why doesn't the following work even though it looks like it should?

In [ ]:
# df.loc[df.loc[df['carat'] < 5] & df.loc[df['cut'] == 'Ideal']]

Inside the `df.loc[]`

We have `df.loc[df['carat'] < 5] & df.loc[df['cut'] == 'Ideal']`

Each of those is a `DataFrame`

Pandas is trying to filter on a Boolean Series (think one column)

What that code above is trying to do is filter a DataFrame with *another* DataFrame. 

Pandas doesn't like that!

---

## Sorting

We can sort individual Series or sort the entire DataFrame

In [ ]:
# Or the entire dataframe


In [ ]:
# We can sort by more than one column


---

## Groupby

What if we want summary statistics with respect to some categorical variable? For example, the price of a diamond probably varies widely between different diamond cuts. To tackle this problem, we'll use the Split-Apply-Combine technique.

* Split: Separate your data into different DataFrames, one for each category.
* Apply: On each split-up DataFrame, apply some function or transformation (for example, the mean).
* Combine: Take the results and combine the split-up DataFrames back into one aggregate DataFrame.
This might sound complicated, but it's actually only two commands in pandas (the Combine step is done for us).

In [ ]:
# What is the mean price by diamond cut?


In [ ]:
# Groupby the cut, then avg price PER each cut


In [ ]:
# Can even call .describe() on this!


---

## Adding, Dropping, Renaming Columns

In [ ]:
# To create a new col
# df['new_col'] = values



In [ ]:
# How can we drop a column? .drop(columns = ['col_to_drop'])


In [ ]:
# We need to assign this output to a variable


In [ ]:
# Let's do a derived column for x, y, z



In [ ]:
# Let's rename x, y, z


---

## Export our dataset

Export to the `/data/processed` folder.

# 📊 Pandas Checklist: `[]` vs `.loc[]`

---

## 1. Use `df['col']` when...

- **Select one column**
```python
df['price']
```

- **Build a boolean mask / filter condition**
```python
over_10k = df['price'] > 10_000
ideal_cut = df['cut'] == 'Ideal'
```

- **Create or replace an entire column**
```python
df['price'] = df['price'].astype(float)
```

- **Perform simple column operations**
```python
avg_price = df['price'].mean()
```

---

## 2. Use `df[['col1', 'col2']]` when...

- **Select multiple columns**
```python
df[['price', 'carat']]
```

> **Note:**  
> - Single brackets → `Series`  
> - Double brackets → `DataFrame`

---

## 3. Use `df[mask]` when...

- **Filter rows only**
```python
over_10k = df['price'] > 10_000
df[over_10k]
```

- **Apply multiple conditions**
```python
carat_lt5 = df['carat'] < 5
ideal_cut = df['cut'] == 'Ideal'

df[carat_lt5 & ideal_cut]
```

> ⚠️ This is safe for filtering only.  
> Problems start when you **filter AND assign**.

---

## 4. Use `df.loc[...]` when...

- **Filter rows explicitly**
```python
df.loc[df['price'] > 10_000]
```

- **Filter rows AND select columns**
```python
df.loc[df['price'] > 10_000, 'cut']
df.loc[df['price'] > 10_000, ['cut', 'color', 'clarity']]
```

- **Modify / assign values to part of a DataFrame**
```python
df.loc[df['price'] > 10_000, 'price'] = 10_000
```

- **Update values conditionally**
```python
df.loc[df['cut'] == 'Ideal', 'premium_flag'] = True
```

- **Select rows and columns by label**
```python
df.loc[0:5, ['price', 'carat']]
```

> 🔑 **Big Rule:**  
> If you are **changing part of a DataFrame**, use `.loc[]`

---

## 5. Safe Rule of Thumb

- Use `[]` for:
  - Selecting columns
  - Building masks
  - Simple filtering

- Use `.loc[]` for:
  - Filtering + selecting columns
  - Conditional assignment
  - Modifying subsets of data

---

## 6. Common Good Patterns

```python
# Build masks
over_10k = df['price'] > 10_000
ideal_cut = df['cut'] == 'Ideal'

# Apply mask
expensive_ideal = df.loc[over_10k & ideal_cut]

# Replace entire column
df['price'] = df['price'].astype(float)

# Conditional assignment
df.loc[df['price'] > 10_000, 'price_bucket'] = 'High'
```

---

## 7. Common Bad Patterns

### ❌ Chained indexing
```python
df[df['price'] > 10_000]['price'] = 0
```

**Why it’s bad:**  
Ambiguous—may not modify the original DataFrame.

### ✅ Correct
```python
df.loc[df['price'] > 10_000, 'price'] = 0
```

---

### ❌ Assigning to a filtered DataFrame
```python
filtered = df[df['cut'] == 'Ideal']
filtered['price'] = filtered['price'] * 1.1
```

**Why it’s bad:**  
Can trigger `SettingWithCopyWarning`

### ✅ Correct
```python
df.loc[df['cut'] == 'Ideal', 'price'] = df.loc[df['cut'] == 'Ideal', 'price'] * 1.1
```

---

## 8. Quick Decision Guide

| Task | Syntax |
|------|--------|
| Select one column | `df['col']` |
| Select multiple columns | `df[['col1', 'col2']]` |
| Build a condition | `df['col'] > value` |
| Filter rows | `df[mask]` or `df.loc[mask]` |
| Filter rows + columns | `df.loc[mask, 'col']` |
| Modify subset | `df.loc[mask, 'col'] = value` |
| Replace entire column | `df['col'] = ...` |

---

## 9. Most Important Takeaway

> ✅ Use `[]` to **get data simply**  
> ✅ Use `.loc[]` to **target data precisely**, especially when modifying


---

## NumPy

NumPy is the fundamental package for scientific computing in Python. It is a Python library that provides a multidimensional array object, various derived objects (such as masked arrays and matrices), and an assortment of routines for fast operations on arrays, including mathematical, logical, shape manipulation, sorting, selecting, I/O, discrete Fourier transforms, basic linear algebra, basic statistical operations, random simulation and much more.

[NumPy Docs](https://numpy.org/doc/stable/)

--- 

`Pandas` is built on top of `NumPy`.

Okay, cool? What does that actually mean?

For a lot of our pandas operations, it is actually using NumPy *under the hood*

In [ ]:
# import numpy

---

### Basics of NumPy

In [ ]:
# 1. Create a basic numpy array


* This is a NumPy array (ndarray)
* Think of it as a faster, more powerful list
* Unlike Python lists, NumPy arrays are optimized for numerical operations

In [ ]:
# 2. Create arrays with `arange`


* Similar to Python’s range(), but returns a NumPy array
* Starts at 0, stops before 10

* Third argument = step size
* This gives even numbers from 0 to 8

In [ ]:
# 3. Create evenly spaced values with linspace


* Creates 5 evenly spaced values between 0 and 1
* Useful for plotting, simulations, and continuous ranges

In [ ]:
# 4. Random number generation


* 5 random integers between 1 and 9

* 5 random numbers from a normal distribution (mean=0, std=1)

In [ ]:
# 5. Slicing arrays


* Same slicing syntax as Python lists
* Start index = 2, stop before 6

**Important difference from Python lists!** 

* NumPy slices are views, not copies
* Modifying the slice changes the original array

In [ ]:
# 6. Vectorized Operations


* Multiplies every element by 2
* No loops required!

Works element-wise across the entire array, this is called **vectorization**

In [ ]:
# Why does this matter? Bridging back to pandas, read in diamonds dataset again


* This works because pandas uses NumPy under the hood
* You've been using NumPy, you just didn't realize it yet!

---

## Takeaway

NumPy is a very powerful library for mathematical operations. Pandas is built on top of numpy to take advantage of NumPy's speed. 

NumPy is much more efficient for storing data, vectorized math (no loops).

Think of pandas as the spreadsheet with labels. NumPy is the calculation engine behind the spreadsheet.